# California Housing Dataset
This is the Part 2 of the California Housing Dataset Project. Here we would use Models like Decision Tree Regressor, Random Forest Regressor and Gradient Boosting Regressor to predict the housing prices in California. We would also evaluate the performance of these models using metrics like Mean Absolute Error (MAE) and R-squared (R²).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

### Loading the Dataset

In [3]:
# Loading the Data 
df = pd.read_csv('housing.csv')
print("The Dataset has {} rows and {} columns.".format(df.shape[0], df.shape[1]))

The Dataset has 20640 rows and 10 columns.


# Understanding the Dataset


In [4]:
# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# Summary statistics
df.describe()


Columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Data types:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


### Getting Rid of the Missing Values of the Dataset

In [5]:
# We could have also added values by mean or median but since this is such a small percentage of the data, we can drop these rows without losing much information.
df.dropna(inplace=True)
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0])
if missing.sum() == 0:
    print("\nAll missing values have been handled.")

Series([], dtype: int64)

All missing values have been handled.


## Corelation of Data 

In [ ]:
corr_with_target = df.select_dtypes(include="number").corr()["median_house_value"].sort_values(ascending=False)
print(corr_with_target)
# Since we have done the same in the previous part, we can skip the EDA part and move on to the modeling part.


median_house_value    1.000000
median_income         0.688355
total_rooms           0.133294
housing_median_age    0.106432
households            0.064894
total_bedrooms        0.049686
population           -0.025300
longitude            -0.045398
latitude             -0.144638
Name: median_house_value, dtype: float64


## Splitting the Data into Training and Testing Sets

In [7]:
# Spliting the data into training and testing sets
target = "median_house_value"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
# identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns  # should include ocean_proximity
print("Numeric features:", numeric_features.tolist())
print("Categorical features:", categorical_features.tolist())

Numeric features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical features: ['ocean_proximity']


# Preprocess blocks
These are simply to deal with the missing values and to split the data into training and testing sets. We would be using the same code as we used in the previous part of the project.


In [9]:
numeric_transformer = Pipeline(steps=[ # Numeric pipeline
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[ # Categorical pipeline
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Model Training 

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Evaluation Metrics are being defined here so that we can use them for all the models we will train.

def eval_model(model, X_test, y_test, name="model"):
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    print(f"{name} -> RMSE: {rmse:.2f} | MAE: {mae:.2f}")

print("Evaluation metrics have been defined. We can use them to evaluate our models.")

Evaluation metrics have been defined. We can use them to evaluate our models.


## Decision Tree (Regression) — simple explanation

A **Decision Tree** predicts a house price by asking a sequence of simple questions and splitting the data into smaller groups.

### How it works (house price example)
Imagine we want to predict a house price. The tree might ask questions like:
1. **Is the area (sq ft) > 1500?**
2. If yes: **Is the location “good”?**
3. If yes: **Is the house newer than 10 years?**

Each question sends the house to a **branch** (like a flowchart).
At the end of a branch (called a **leaf**), the model predicts the price as the **average price of the training houses** that ended up in that same leaf.

### Why it’s useful
- It can learn **non‑linear patterns** (price doesn’t always increase in a straight line).
- It’s easy to *visualize as rules* (a set of if/else decisions).

### Main downside (important)
A single decision tree can **overfit**:
- It may learn the training data “too perfectly”
- Then it performs worse on new unseen houses

### Common fix
Limit the tree complexity (example settings):
- `max_depth`: limits how many questions it can ask
- `min_samples_leaf`: forces each final group to have enough houses

### One-line summary
A Decision Tree is a **rule-based flowchart** that predicts a number by repeatedly splitting data into smaller, more similar groups.


In [12]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

tree = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeRegressor(random_state=42))
])

tree.fit(X_train, y_train)
eval_model(tree, X_test, y_test, "Decision Tree")

Decision Tree -> RMSE: 67458.60 | MAE: 43036.13


## Random Forest (Regression) — simple explanation

A **Random Forest** is a model that predicts by using **many decision trees together** instead of just one.

### How it works (house price example)
- Imagine you train **300 different decision trees**.
- Each tree learns from a slightly different “view” of the data (and doesn’t look at all features in the same way).
- Each tree gives its own house price prediction.
- The Random Forest final prediction is the **average of all tree predictions**.

### Why it usually beats a single Decision Tree
A single tree can be too confident and “memorize” the training data (overfit).
A forest averages many trees, so extreme wrong guesses get “smoothed out”.

### Pros
- Often **more accurate** than a single decision tree
- Handles **complex patterns** (non-linear relationships)
- More stable predictions

### Cons
- Less easy to explain than one tree (because it’s many trees)
- Can be slower to train (but usually worth it)

### One-line summary
A Random Forest predicts by **averaging the results of many decision trees**, which usually makes it more accurate and less overfitted.

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

rf = Pipeline(steps=[ # This is a more controlled version of the Random Forest Regressor where we are setting some hyperparameters to prevent overfitting.
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
eval_model(rf, X_test, y_test, "Random Forest (controlled)")

Random Forest (controlled) -> RMSE: 51240.62 | MAE: 34006.11


## Gradient Boosting (Regression) — simple explanation

**Gradient Boosting** predicts by building **many small decision trees one after another**, where each new tree focuses on fixing the mistakes made so far.

### How it works (house price example)
1. First small tree makes a rough guess for every house.
2. We look at the **errors** (how far off the predictions are).
3. The next tree learns patterns in those errors (example: “big houses are being under-priced”).
4. We add that correction to the previous prediction.
5. Repeat this many times.

So the final prediction is like:
**final prediction = first guess + small fixes + small fixes + ...**

### Why it’s powerful
- It’s great at learning **complex patterns**
- Often gives **top performance** on tabular data (like house prices)

### Things to be careful about
- If you boost too aggressively, it can overfit.
- Common “control knobs”:
  - `n_estimators`: how many correction steps (trees)
  - `learning_rate`: how big each correction is (smaller is usually safer)
  - `max_depth`: how complex each small tree is (usually kept small)

### One-line summary
Gradient Boosting makes predictions by **improving step-by-step**, adding many small trees that each fix previous mistakes.

In [14]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

gbr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", GradientBoostingRegressor(
        n_estimators=500,     # number of boosting steps (trees)
        learning_rate=0.05,   # smaller steps, usually more stable
        max_depth=3,          # small trees are typical for boosting
        random_state=42
    ))
])

gbr.fit(X_train, y_train)
eval_model(gbr, X_test, y_test, "Gradient Boosting (tuned)")

Gradient Boosting (tuned) -> RMSE: 51799.53 | MAE: 35227.55


# Report For Evaluation Metrics
# Report — Regression (Tree-Based Models)

Goal: **Understand ensemble methods** and compare tree models with earlier regression.

---

## Results Summary (Lower is better for RMSE and MAE)

| Model | RMSE | MAE |
|------|------:|----:|
| **Random Forest (controlled)** | **51,240.62** | **34,006.11** |
| Gradient Boosting (tuned) | 51,799.53 | 35,227.55 |
| Decision Tree | 67,458.60 | 43,036.13 |
| Poly(deg=2) + Ridge(alpha=10) | 65,212.81 | 46,612.52 |

---

## Comparison Notes (simple wording)

### Polynomial Regression vs Tree Models
- **Poly + Ridge** tries to fit a “curved line” style relationship.
- In this dataset, **tree ensembles** (Random Forest / Gradient Boosting) did better because they can handle **more complex patterns** without needing us to manually choose polynomial features.

### Decision Tree vs Random Forest vs Gradient Boosting
- **Decision Tree** (single tree) performed the worst among the tree models.  
  Reason: one tree can “memorize” training patterns and then miss on new data.
- **Random Forest** performed the best overall.  
  Reason: it averages many trees → more stable, fewer extreme wrong guesses.
- **Gradient Boosting** was very close to Random Forest.  
  Reason: it improves step-by-step by fixing earlier mistakes, often strong but can be sensitive to settings.

---

## Best Model (based on lowest RMSE and MAE)
✅ **Random Forest (controlled)**  
- RMSE: **51,240.62**  
- MAE: **34,006.11**

This is the best among the tree algorithms and also best overall in this comparison.

---

## Key Learning 
**Ensemble methods** (Random Forest / Gradient Boosting) often beat single models (like one Decision Tree or polynomial regression) because combining many trees reduces errors and handles complex data better.

## Getting the Possible Best Model

In [15]:
import joblib # saving the model of Random Forest Regressor since it is the best model among the three we trained.
joblib.dump(rf, "california_housing_best_tree_model.joblib")
print("The best model has been saved as 'california_housing_best_tree_model.joblib'.")

The best model has been saved as 'california_housing_best_tree_model.joblib'.
